# 02 — Engenharia de Features Anti-Leakage, Experimentos e Baseline Heurístico
**Hipótese Científica:** Features longitudinais construídas estritamente no instante $t$ (com particionamento temporal *out-of-time* $t \rightarrow t+1$) superam uma regra heurística simples na identificação preventiva de necessidade de acompanhamento, sem vazamento de informação futura nem superajuste (*overfitting*) à janela histórica.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline.features import FEATURE_COLUMNS, build_longitudinal_datasets
from src.pipeline.train import run_training_pipeline

splits, cohort_reports = build_longitudinal_datasets()
feature_cols = FEATURE_COLUMNS
print(f"Quantidade de features de entrada: {len(feature_cols)}")
print("Features:", feature_cols)
for k, v in splits.items():
    print(f"Split {k}: shape={v.shape} | target_rate={v['target'].mean():.2%}")


C:\tech-challenge-fase5-grupo\.venv\Lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


C:\tech-challenge-fase5-grupo\.venv\Lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_api" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Quantidade de features de entrada: 24
Features: ['fase_num', 'idade', 'anos_na_ong', 'defasagem', 'inde', 'iaa', 'ieg', 'ips', 'ida', 'ipv', 'ian', 'mat', 'por', 'ing', 'tem_nota_ing', 'pedra_ord', 'pedra_anterior_ord', 'delta_pedra_1y', 'delta_inde_1y', 'gap_iaa_ida', 'gap_ieg_ida', 'media_psico', 'genero_bin', 'escola_publica']
Split train: shape=(570, 27) | target_rate=20.70%
Split val: shape=(678, 27) | target_rate=17.70%
Split test: shape=(594, 27) | target_rate=20.03%


## 1. Integridade Temporal dos Splits e Taxa-Base do Alvo ($t+1$)
Auditoria do volume de amostras e da prevalência da classe positiva (`target_acompanhamento_t1`) em treino, validação e teste fora do tempo.

In [2]:
resumo_splits = []
for nome_split, df_split in splits.items():
    target = df_split["target"]
    resumo_splits.append({
        "split": nome_split,
        "linhas": len(df_split),
        "colunas": df_split.shape[1],
        "positivos": int(target.sum()),
        "taxa_positiva": round(float(target.mean()), 4),
        "nulos_em_features": int(df_split[feature_cols].isna().sum().sum()),
    })
display(pd.DataFrame(resumo_splits))

,split,linhas,colunas,positivos,taxa_positiva,nulos_em_features
0,train,570,27,118,0.2070,675
1,val,678,27,120,0.1770,981
2,test,594,27,119,0.2003,323


## 2. Verificação Formal de Anti-Leakage e Esquema
Garantia de que variáveis de desfecho, identificadores diretos ou atributos exclusivos de $t+1$ não contaminam o vetor de features usado no treino e na inferência.

In [3]:
termos_suspeitos = ("target", "t1", "t_plus_1", "futuro", "label")
vazamentos = [c for c in feature_cols if any(t in c.lower() for t in termos_suspeitos)]
assert not vazamentos, f"Possível leakage detectado em feature_cols: {vazamentos}"
for nome_split, df_split in splits.items():
    faltantes = [c for c in feature_cols if c not in df_split.columns]
    assert not faltantes, f"Colunas ausentes em {nome_split}: {faltantes}"
print("Auditoria anti-leakage e consistência de esquema aprovadas em todos os splits.")

Auditoria anti-leakage e consistência de esquema aprovadas em todos os splits.


## 3. Execução do Pipeline de Treino e Comparação com Baseline
Acionamento de `run_training_pipeline()` para treinar os candidatos, avaliar frente ao baseline heurístico e registar métricas/artefatos.

In [4]:
results, gate_report = run_training_pipeline()
rows = []
for m_name, m_info in results.items():
    val_m = m_info.get("val", {})
    oot_m = m_info.get("test_oot", {})
    rows.append({
        "modelo": m_name,
        "val_pr_auc": getattr(val_m, "pr_auc", val_m.get("pr_auc") if isinstance(val_m, dict) else None),
        "val_roc_auc": getattr(val_m, "roc_auc", val_m.get("roc_auc") if isinstance(val_m, dict) else None),
        "val_f2": getattr(val_m, "f2", val_m.get("f2") if isinstance(val_m, dict) else None),
        "val_recall": getattr(val_m, "recall", val_m.get("recall") if isinstance(val_m, dict) else None),
        "oot_pr_auc": getattr(oot_m, "pr_auc", oot_m.get("pr_auc") if isinstance(oot_m, dict) else None),
        "oot_recall": getattr(oot_m, "recall", oot_m.get("recall") if isinstance(oot_m, dict) else None),
    })
display(pd.DataFrame(rows))
print("Quality Gate aprovado:", gate_report.get("approved"), "| Campeão:", gate_report.get("champion_model"))


2026/11/24 07:24:23 INFO mlflow.tracking.fluent: Experiment with name 'passos-magicos-mlops-fase5' does not exist. Creating a new experiment.


2026/11/24 07:24:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/11/24 07:24:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/11/24 07:24:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


C:\tech-challenge-fase5-grupo\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] O sistema não pode encontrar o arquivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\tech-challenge-fase5-grupo\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Usuario\A

2026/11/24 07:24:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


,modelo,val_pr_auc,val_roc_auc,val_f2,val_recall,oot_pr_auc,oot_recall
0,baseline_heuristic,0.4207,0.7861,0.5495,0.5833,0.2243,0.2941
1,baseline_logistic_regression,0.4903,0.8172,0.6448,0.8500,0.2250,0.5798
2,candidate_random_forest,0.4994,0.8220,0.6564,0.7833,0.2298,0.4622
3,candidate_hist_gradient_boosting,0.4725,0.8064,0.6440,0.7750,0.2209,0.4454


Quality Gate aprovado: True | Campeão: candidate_random_forest


## 4. Rigor Científico: Métricas, Overfitting e Limitações
* **Escolha de Métricas:** Em cenários com desbalanceamento de classes e custo assimétrico de falso negativo (deixar de sinalizar um estudante que precisará de acompanhamento), *PR-AUC*, *ROC-AUC*, *Recall* e *F2-Score* são mais informativos que acurácia global.
* **Risco de Overfitting e Domain Shift:** Modelos baseados em árvores podem superajustar padrões específicos de 2022–2023 se não forem validados em janelas posteriores (*out-of-time*) e calibrados quanto ao limiar de decisão.
* **Estratégia de Validação:** Divisão cronológica estrita com verificação de estabilidade temporal antes de qualquer promoção via *Quality Gate*.